In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#import yfinance as yf

#TA-Lib for indicators
import talib
import pynance as py
import quantstats as qs


## ○ Use additional finance data
## ○ Load and prepare the data.


#### ■ Load your stock price data into a pandas DataFrame. Ensure your data includes columns like Open, High, Low, Close, and Volume.

In [13]:
# 1. Define the list of tickers and the structure to hold the DataFrames
tickers = ['NVDA', 'AAPL', 'AMZN', 'GOOG', 'META', 'MSFT']
file_extension = ".csv"
stock_data = {}

##### loading all tickers on one stock_data dictionary. so can be access stock_data['NVDA']

In [14]:
# 2. Loop through the tickers and load the data
for ticker in tickers:
    # Construct the expected file name (e.g., 'NVDA.csv')
    filename = "../data/" +ticker + file_extension
    
    try:
        # Load the CSV file:
        # - index_col='Date': Assumes your date column is named 'Date'
        # - parse_dates=True: Converts the date column into datetime objects
        df = pd.read_csv(
            filename, 
            index_col='Date', 
            parse_dates=True
        )
        
        # Store the DataFrame in the dictionary
        stock_data[ticker] = df
        
        print(f" Loaded {filename}. Shape: {df.shape}")
        
    except FileNotFoundError:
        print(f" ERROR: File '{filename}' not found. Check your file path.")
    except Exception as e:
        print(f" ERROR processing {filename}: {e}")


# 3. Verification
print("\n--- Summary ---")
print("Successfully loaded DataFrames for:", list(stock_data.keys()))

if 'NVDA' in stock_data:
    print("\nFirst 5 rows of NVDA data:")
    print(stock_data['NVDA'].head())

 Loaded ../data/NVDA.csv. Shape: (3774, 5)
 Loaded ../data/AAPL.csv. Shape: (3774, 5)
 Loaded ../data/AMZN.csv. Shape: (3774, 5)
 Loaded ../data/GOOG.csv. Shape: (3774, 5)
 Loaded ../data/META.csv. Shape: (2923, 5)
 Loaded ../data/MSFT.csv. Shape: (3774, 5)

--- Summary ---
Successfully loaded DataFrames for: ['NVDA', 'AAPL', 'AMZN', 'GOOG', 'META', 'MSFT']

First 5 rows of NVDA data:
               Close      High       Low      Open      Volume
Date                                                          
2009-01-02  0.199652  0.201027  0.184294  0.184982   497124000
2009-01-05  0.203319  0.207904  0.195984  0.197360   705736000
2009-01-06  0.210196  0.216156  0.204695  0.209279   657904000
2009-01-07  0.197589  0.205382  0.190483  0.205382   870096000
2009-01-08  0.192546  0.195067  0.180626  0.195067  1014496000


,Close,High,Low,Open,Volume
Date,,,,,
2009-01-02,0.199652,0.201027,0.184294,0.184982,497124000
2009-01-05,0.203319,0.207904,0.195984,0.197360,705736000
2009-01-06,0.210196,0.216156,0.204695,0.209279,657904000
2009-01-07,0.197589,0.205382,0.190483,0.205382,870096000
2009-01-08,0.192546,0.195067,0.180626,0.195067,1014496000


## ○ Apply Analysis Indicators with TA-Lib
#### ■ You can use TA-Lib to calculate various technical indicators such as moving averages, RSI (Relative Strength Index), and MACD (Moving Average Convergence Divergence
#### Calculated for each ticker

In [18]:


# Define common parameters for the indicators
SMA_SHORT = 20
SMA_LONG = 50
EMA_PERIOD = 20
RSI_PERIOD = 14
FAST_PERIOD = 12
SLOW_PERIOD = 26
SIGNAL_PERIOD = 9

for ticker, df in stock_data.items():
    
    # Check for the essential 'Close' column
    if 'Close' not in df.columns:
        print(f"Skipping {ticker}: 'Close' column not found. Cannot calculate indicators.")
        continue

    # Convert the 'Close' series to a NumPy array for TA-Lib
    close_prices = df['Close'].values

    # --- Moving Averages ---
    
    # 1. Simple Moving Average (20-day)
    df[f'{ticker}_SMA_{SMA_SHORT}'] = talib.SMA(close_prices, timeperiod=SMA_SHORT)
    
    # 2. Simple Moving Average (50-day)
    df[f'{ticker}_SMA_{SMA_LONG}'] = talib.SMA(close_prices, timeperiod=SMA_LONG)
    
    # 3. Exponential Moving Average (20-day)
    df[f'{ticker}_EMA_{EMA_PERIOD}'] = talib.EMA(close_prices, timeperiod=EMA_PERIOD)

    # --- Momentum Indicators (Recalculating/Confirming) ---
    
    # 4. RSI (14-day)
    df[f'{ticker}_RSI_{RSI_PERIOD}'] = talib.RSI(close_prices, timeperiod=RSI_PERIOD)
    
    # 5. MACD (and its components)
    macd, macd_signal, macd_hist = talib.MACD(
        close_prices, 
        fastperiod=FAST_PERIOD, 
        slowperiod=SLOW_PERIOD, 
        signalperiod=SIGNAL_PERIOD
    )
    
    # Add MACD components as separate columns
    df[f'{ticker}_MACD'] = macd
    df[f'{ticker}_MACD_Signal'] = macd_signal
    df[f'{ticker}_MACD_Hist'] = macd_hist
    
    print(f"✅ Calculated all 5 indicators for {ticker}.")

print("\nAll stock DataFrames in 'stock_data' have been updated with new technical indicators.")

✅ Calculated all 5 indicators for NVDA.
✅ Calculated all 5 indicators for AAPL.
✅ Calculated all 5 indicators for AMZN.
✅ Calculated all 5 indicators for GOOG.
✅ Calculated all 5 indicators for META.
✅ Calculated all 5 indicators for MSFT.

All stock DataFrames in 'stock_data' have been updated with new technical indicators.


In [19]:
# Check the columns for Google (GOOG)
print("\nColumns of GOOG DataFrame after calculation:")
print(stock_data['GOOG'].columns)

# Check the last few rows to see non-NaN indicator values
print("\nLast 10 rows of GOOG data with new indicators:")
print(stock_data['GOOG'].tail(10)[['Close', 'GOOG_SMA_20', 'GOOG_RSI_14', 'GOOG_MACD_Hist']])


Columns of GOOG DataFrame after calculation:
Index(['Close', 'High', 'Low', 'Open', 'Volume', 'GOOG_SMA_20', 'GOOG_SMA_50',
       'GOOG_EMA_20', 'GOOG_RSI_14', 'GOOG_MACD', 'GOOG_MACD_Signal',
       'GOOG_MACD_Hist'],
      dtype='object')

Last 10 rows of GOOG data with new indicators:
                 Close  GOOG_SMA_20  GOOG_RSI_14  GOOG_MACD_Hist
Date                                                            
2023-12-15  132.930344   134.624729    48.205282       -0.253484
2023-12-18  136.257538   134.637142    55.322365       -0.012916
2023-12-19  137.161362   134.646081    57.048922        0.195828
2023-12-20  138.710785   134.697729    59.909189        0.414435
2023-12-21  140.836227   134.786124    63.499976        0.663581
2023-12-22  141.750000   135.009596    64.953309        0.839969
2023-12-26  141.849304   135.246477    65.115870        0.907615
2023-12-27  140.478683   135.386520    60.916026        0.807682
2023-12-28  140.319748   135.628861    60.429303        0.6